Image_based Training Pipeline
Multi-Component Classification with Hyperparameter Tuning
This script implements a comprehensive Deep Learning (DL) pipeline using a Convolutional Neural Network (CNN) to classify seismic signals using their images.

1. Prerequisites & Inputs
To run this pipeline, ensure the following files and folders are in your working directory:

Metadata (Pickle Files):

tr_No_Fil.pkl (Training set)

val_No_Fil.pkl (Validation set)

te_No_Fil.pkl (Test set)

Seismic Images (300x1200 px):

train_IM_c123/

val_IM_c123/

test_IM_c123/ (c123 means it contains all components)

Naming convention: {component}_{name}.png (e.g., x_1234.png)

2. Core Functionality
A. Model Architecture
SeismicCNN: A 5-layer 2D CNN optimized for high-aspect-ratio seismic plots.

Dynamic Flattening: Automatically adapts to input image dimensions.

Tunable Head: Includes Dropout and Sigmoid output for binary classification.

B. Data Balancing (SMOTE)
The script applies SMOTE (Synthetic Minority Over-sampling Technique) to the training data. This ensures the model learns equally from both high_quality and low_quality ground motion (GM) waveforms, preventing class bias.

C. Sequential Hyperparameter Tuning
The pipeline optimizes the following parameters one by one:

learning_rate

dropout_rate

weight_decay

batch_size
It locks the best value for each before moving to the next, ensuring stable convergence.

3. Outputs & Evaluation
Model Weights: Saved as best_seismic_IM_model.pth.

Confusion Matrix: A visual heatmap showing True/False Positives and Negatives.

Metrics: Detailed Classification Report including Accuracy, Precision, Recall, and F1-Score.

4. Execution
Run the cell below to start the tuning process. The final evaluation will trigger automatically once the best model is found.


In [ ]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
import shutil

# load data
tr_No_Fil = pd.read_pickle('tr_No_Fil.pkl')
te_No_Fil = pd.read_pickle('te_No_Fil.pkl')
val_No_Fil = pd.read_pickle('val_No_Fil.pkl')

# ====================================================================
# STEP 1: MODEL DEFINITION (ADAPTIVE & TUNABLE)
# ====================================================================

class SeismicCNN(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(SeismicCNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.BatchNorm2d(16),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.BatchNorm2d(32),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.BatchNorm2d(64),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.BatchNorm2d(128),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.BatchNorm2d(256)
        )
        
        self._to_linear = None
        self._get_conv_output_size()
        
        self.fc_layers = nn.Sequential(
            nn.Linear(self._to_linear, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def _get_conv_output_size(self):
        with torch.no_grad():
            # Based on your target_size = (1200, 300) -> (Batch, Channel, Height, Width)
            x = torch.randn(1, 1, 300, 1200)
            out = self.conv_layers(x)
            self._to_linear = out.shape[1] * out.shape[2] * out.shape[3]

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        return self.fc_layers(x)

# ====================================================================
# STEP 2: DATA PROCESSING & DATASET CLASSES
# ====================================================================

def load_combined_images_for_smote(dataframe, img_dir):
    print(f"Loading images from {img_dir} for SMOTE...")
    images, labels = [], []
    transform = transforms.Compose([transforms.Grayscale(1), transforms.ToTensor()])
    
    for idx in range(len(dataframe)):
        row = dataframe.iloc[idx]
        for comp in ['x', 'y', 'z']:
            score_col = f'score_{comp}'
            if score_col in row:
                img_path = os.path.join(img_dir, f"{comp}_{row['name']}.png")
                if os.path.exists(img_path):
                    img = Image.open(img_path)
                    images.append(transform(img))
                    labels.append(row[score_col])
    return torch.stack(images), torch.tensor(labels, dtype=torch.float32)

def apply_smote_to_combined_images(images, labels):
    print("Applying SMOTE...")
    b, c, h, w = images.shape
    images_flat = images.view(b, -1).numpy()
    smote = SMOTE(random_state=42)
    img_res, lbl_res = smote.fit_resample(images_flat, labels.numpy())
    img_res = torch.FloatTensor(img_res.reshape(-1, c, h, w))
    lbl_res = torch.FloatTensor(lbl_res)
    return img_res, lbl_res

class CombinedSeismicDataset(Dataset):
    def __init__(self, dataframe, img_dir):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transforms.Compose([transforms.Grayscale(1), transforms.ToTensor()])
        self.data = []
        for idx in range(len(dataframe)):
            row = dataframe.iloc[idx]
            for comp in ['x', 'y', 'z']:
                if f'score_{comp}' in row:
                    self.data.append({'img': f"{comp}_{row['name']}.png", 'lbl': row[f'score_{comp}']})
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return self.transform(Image.open(os.path.join(self.img_dir, item['img']))), torch.tensor(item['lbl'], dtype=torch.float32)

class SMOTECombinedDataset(Dataset):
    def __init__(self, images, labels):
        self.images, self.labels = images, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

# ====================================================================
# STEP 3: TUNING ENGINE
# ====================================================================

def train_single_config(train_loader, val_loader, config, device, epochs=15):
    model = SeismicCNN(dropout_rate=config['dropout_rate']).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    
    best_loss = float('inf')
    best_state = None
    
    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs).squeeze(), labels)
            loss.backward()
            optimizer.step()
            
        model.eval()
        v_loss = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                v_loss += criterion(model(inputs).squeeze(), labels).item()
        
        avg_v_loss = v_loss / len(val_loader)
        if avg_v_loss < best_loss:
            best_loss = avg_v_loss
            best_state = model.state_dict()
            
    return best_loss, best_state

# ====================================================================
# MAIN WORKFLOW
# ====================================================================

def run_tuned_combined_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Hyperparameter Grids
    param_grids = {
        'learning_rate': [0.1, 0.01, 0.001, 0.0001],
        'dropout_rate': [0.2, 0.3, 0.5, 0.7],
        'weight_decay': [0.0, 1e-5, 1e-4, 1e-3],
        'batch_size': [16, 32, 64, 128]
    }
    current_best_config = {'learning_rate': 0.001, 'dropout_rate': 0.5, 'weight_decay': 0.0, 'batch_size': 32}
    final_best_state = None

    # 2. Data Preparation (Assumes tr_No_Fil and val_No_Fil are in global scope)
    # Note: Ensure images exist in these directories first!
    train_imgs, train_lbls = load_combined_images_for_smote(tr_No_Fil, "train_IM_c123")
    train_res_imgs, train_res_lbls = apply_smote_to_combined_images(train_imgs, train_lbls)
    val_dataset = CombinedSeismicDataset(val_No_Fil, "val_IM_c123")

    # 3. Sequential Tuning
    for param, values in param_grids.items():
        print(f"\n--- Tuning {param} ---")
        best_param_loss = float('inf')
        
        for val in values:
            test_config = current_best_config.copy()
            test_config[param] = val
            print(f"Testing {param}={val}...")
            
            t_loader = DataLoader(SMOTECombinedDataset(train_res_imgs, train_res_lbls), 
                                  batch_size=test_config['batch_size'], shuffle=True)
            v_loader = DataLoader(val_dataset, batch_size=test_config['batch_size'])
            
            loss, state = train_single_config(t_loader, v_loader, test_config, device)
            
            if loss < best_param_loss:
                best_param_loss = loss
                current_best_config[param] = val
                final_best_state = state 

    # 4. Save Final Output
    if final_best_state:
        torch.save(final_best_state, 'best_seismic_IM_model.pth')
        print("\nOptimization Complete.")
        print("Final Optimal Configuration:", current_best_config)
        return current_best_config
    else:
        print("Error: No best state found.")

if __name__ == "__main__":
    # Define device globally within main to be used by all steps
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Start Tuning and Training and capture the returned config
    # This prevents the "variable scope" warning
    best_config = run_tuned_combined_training()

    # Check if training was successful before proceeding
    if best_config is not None:
        # ==========================================================
        # FINAL EVALUATION
        # ==========================================================
        print("\n" + "="*40)
        print("STARTING FINAL EVALUATION ON TEST SET")
        print("="*40)

        # Load test labels/metadata
        test_ds = CombinedSeismicDataset(te_No_Fil, "test_IM_c123") 
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

        # Use the captured config instead of the internal function variable
        best_dropout = best_config['dropout_rate']
        
        # Re-initialize and load the best saved model
        model = SeismicCNN(dropout_rate=best_dropout).to(device)
        model.load_state_dict(torch.load('best_seismic_IM_model.pth'))
        model.eval()

        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                outputs = model(inputs).squeeze()
                # Ensure outputs and labels have compatible shapes for comparison
                preds = (outputs > 0.5).float().cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())

        # Plot Confusion Matrix
        cm = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=['Bad', 'Good'], yticklabels=['Bad', 'Good'])
        plt.title(f'Final Test Results (Best Dropout: {best_dropout})')
        plt.ylabel('Actual Label')
        plt.xlabel('Predicted Label')
        plt.show()

        # Classification Metrics
        print(f"Final Test Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
        print("\nDetailed Classification Report:\n")
        print(classification_report(all_labels, all_preds))
    else:
        print("Optimization did not return a valid configuration. Skipping evaluation.")

Loading images from train_IM_c123 for SMOTE...
Applying SMOTE...
